In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import os

DRIVE_ROOT = "/content/drive/MyDrive/msc-deepfake"
LOCAL_HF_CACHE = "/content/hf_cache"

os.environ["HF_HOME"] = LOCAL_HF_CACHE
os.environ["HF_HUB_CACHE"] = LOCAL_HF_CACHE
os.environ["TRANSFORMERS_CACHE"] = LOCAL_HF_CACHE
os.makedirs(LOCAL_HF_CACHE, exist_ok=True)

OUT_FOLDER = f"{DRIVE_ROOT}/generated_videos/wan"
METADATA_FOLDER = f"{DRIVE_ROOT}/metadata/wan"
PROMPTS_FILE = f"{DRIVE_ROOT}/batch_prompts/week2_prompts_v1.txt"

os.makedirs(OUT_FOLDER, exist_ok=True)
os.makedirs(METADATA_FOLDER, exist_ok=True)

print(f"OUT_FOLDER: {OUT_FOLDER}")

import subprocess
result = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'],
                       capture_output=True, text=True)
print(f"\nGPU: {result.stdout.strip()}")

OUT_FOLDER: /content/drive/MyDrive/msc-deepfake/generated_videos/wan

GPU: NVIDIA A100-SXM4-80GB, 81920 MiB


In [3]:
!pip install -q --upgrade diffusers transformers accelerate imageio imageio-ffmpeg sentencepiece ftfy einops safetensors
print("Installed.")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.6/11.6 MB 141.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 5.0 MB/s eta 0:00:00
Installed.


In [4]:
from diffusers import WanPipeline
import torch, gc

gc.collect()
torch.cuda.empty_cache()

print("Loading Wan 2.2 T2V-A14B on A100...")
print("First-time download is ~30-40 GB and takes 15-25 minutes.\n")

pipe = WanPipeline.from_pretrained(
    "Wan-AI/Wan2.2-T2V-A14B-Diffusers",
    torch_dtype=torch.bfloat16
)

pipe.to("cuda")
pipe.vae.enable_tiling()
pipe.vae.enable_slicing()

print("Wan 2.2 T2V-A14B loaded.")
print(f"GPU memory used: {torch.cuda.memory_allocated()/1e9:.2f} GB")
print(f"GPU total: {torch.cuda.get_device_properties(0).total_memory/1e9:.2f} GB")

Flax classes are deprecated and will be removed in Diffusers v0.40.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.
Flax classes are deprecated and will be removed in Diffusers v0.40.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.


Loading Wan 2.2 T2V-A14B on A100...
First-time download is ~30-40 GB and takes 15-25 minutes.



model_index.json:   0%|          | 0.00/498 [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 41 files:   0%|          | 0/41 [00:00<?, ?it/s]

Loading pipeline components...:   0%|          | 0/6 [00:00<?, ?it/s]

Loading checkpoint shards:   0%|          | 0/12 [00:00<?, ?it/s]

Loading checkpoint shards:   0%|          | 0/12 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/242 [00:00<?, ?it/s]

Wan 2.2 T2V-A14B loaded.
GPU memory used: 69.06 GB
GPU total: 85.09 GB


In [13]:
import datetime, json
from diffusers.utils import export_to_video

CINEMATIC_SUFFIX = ", natural lighting, cinematic quality, high detail, realistic"
NEGATIVE_PROMPT = "blurry, low quality, distorted, unnatural, deformed"

GEN_PARAMS = {
    "model_id": "Wan-AI/Wan2.2-T2V-A14B-Diffusers",
    "height": 480,
    "width": 832,
    "num_frames": 49,               # was 81 — Wan's non-native
    "num_inference_steps": 25,
    "guidance_scale": 5.0,
    "seed": 42,
    "fps": 16,
    "torch_dtype": "bfloat16",
    "cinematic_suffix": CINEMATIC_SUFFIX,
    "negative_prompt": NEGATIVE_PROMPT,
}

def generate_and_save(prompt_id, category, prompt_text, out_folder, metadata_folder):
    full_prompt = prompt_text + CINEMATIC_SUFFIX
    print(f"\n[{prompt_id}] {category}: {prompt_text[:60]}...")

    start = datetime.datetime.now()
    generator = torch.Generator(device="cuda").manual_seed(GEN_PARAMS["seed"])

    video = pipe(
        prompt=full_prompt,
        negative_prompt=NEGATIVE_PROMPT,
        height=GEN_PARAMS["height"],
        width=GEN_PARAMS["width"],
        num_frames=GEN_PARAMS["num_frames"],
        num_inference_steps=GEN_PARAMS["num_inference_steps"],
        guidance_scale=GEN_PARAMS["guidance_scale"],
        generator=generator,
    ).frames[0]

    duration = (datetime.datetime.now() - start).total_seconds()
    timestamp = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
    filename = f"{prompt_id}_wan_{timestamp}"
    video_path = f"{out_folder}/{filename}.mp4"
    metadata_path = f"{metadata_folder}/{filename}.json"

    # Wan native fps is 16
    export_to_video(video, video_path, fps=GEN_PARAMS["fps"])

    metadata = {
        "video_id": filename,
        "prompt_id": prompt_id,
        "category": category,
        "original_prompt": prompt_text,
        "full_prompt": full_prompt,
        "generator": "Wan 2.2 T2V-A14B",
        "generation_time_seconds": duration,
        "generation_datetime": timestamp,
        "gpu": "A100",
        **GEN_PARAMS,
    }
    with open(metadata_path, "w") as f:
        json.dump(metadata, f, indent=2)

    print(f"Saved: {video_path} ({duration:.1f}s = {duration/60:.1f}min)")
    return video_path

print("Function defined with parameter-dict logging.")

Function defined with parameter-dict logging.


In [8]:
with open(PROMPTS_FILE) as f:
    lines = [line.strip() for line in f
             if line.strip() and not line.strip().startswith("#") and "|" in line]

test_line = lines[0]
parts = test_line.split("|", 2)
prompt_id, category, prompt_text = parts

print(f"Sanity check on: {prompt_id}")
generate_and_save(prompt_id, category, prompt_text, OUT_FOLDER, METADATA_FOLDER)

Sanity check on: w2_001

[w2_001] portrait: A woman in her thirties smiling and speaking directly to the...


  0%|          | 0/20 [00:00<?, ?it/s]

Saved: /content/drive/MyDrive/msc-deepfake/generated_videos/wan/w2_001_wan_20260719_155311.mp4 (385.9s = 6.4min)


'/content/drive/MyDrive/msc-deepfake/generated_videos/wan/w2_001_wan_20260719_155311.mp4'

In [12]:
import shutil

FAILED_FOLDER = f"{DRIVE_ROOT}/generated_videos/wan_5sec_FAILED"
os.makedirs(FAILED_FOLDER, exist_ok=True)

moved = 0
for v in os.listdir(OUT_FOLDER):
    if v.endswith(".mp4"):
        shutil.move(f"{OUT_FOLDER}/{v}", f"{FAILED_FOLDER}/{v}")
        moved += 1

for m in os.listdir(METADATA_FOLDER):
    if m.endswith(".json"):
        shutil.move(f"{METADATA_FOLDER}/{m}", f"{FAILED_FOLDER}/{m}")

print(f"Moved {moved} to FAILED_5sec folder")

Moved 3 to FAILED_5sec folder


In [15]:
existing_ids = set()
for v in os.listdir(OUT_FOLDER):
    if v.endswith(".mp4"):
        prompt_id = v.split("_wan_")[0]
        existing_ids.add(prompt_id)

remaining = [line for line in lines if line.split("|")[0] not in existing_ids]

batch = remaining[:8]
print(f"Already done: {len(existing_ids)}")
print(f"Total remaining: {len(remaining)}")
print(f"This batch: {len(batch)}")
print("=" * 60)

successes = []
failures = []

for line in batch:
    parts = line.split("|", 2)
    if len(parts) != 3:
        continue
    prompt_id, category, prompt_text = parts
    try:
        generate_and_save(prompt_id, category, prompt_text, OUT_FOLDER, METADATA_FOLDER)
        successes.append(prompt_id)
    except Exception as e:
        print(f"FAILED on {prompt_id}: {e}")
        failures.append((prompt_id, str(e)))

now_done = len(existing_ids) + len(successes)
print("\n" + "=" * 60)
print(f"Batch complete. Success: {len(successes)}, Failed: {len(failures)}")
print(f"Total Wan videos now: {now_done}/40")
if failures:
    print(f"Failures: {failures}")

Already done: 0
Total remaining: 40
This batch: 8

[w2_001] portrait: A woman in her thirties smiling and speaking directly to the...


  0%|          | 0/25 [00:00<?, ?it/s]

Saved: /content/drive/MyDrive/msc-deepfake/generated_videos/wan/w2_001_wan_20260719_162444.mp4 (244.8s = 4.1min)

[w2_002] portrait: An older man laughing while telling a story, cafe interior, ...


  0%|          | 0/25 [00:00<?, ?it/s]

Saved: /content/drive/MyDrive/msc-deepfake/generated_videos/wan/w2_002_wan_20260719_162850.mp4 (245.3s = 4.1min)

[w2_003] portrait: A young man with glasses reading a book, close-up on his fac...


...

Saved: /content/drive/MyDrive/msc-deepfake/generated_videos/wan/w2_006_wan_20260719_164514.mp4 (245.3s = 4.1min)

[w2_007] hands: A hand pouring milk from a glass bottle into a coffee cup, k...


  0%|          | 0/25 [00:00<?, ?it/s]

Saved: /content/drive/MyDrive/msc-deepfake/generated_videos/wan/w2_007_wan_20260719_164920.mp4 (245.7s = 4.1min)

[w2_008] hands: Someone tying their shoelaces, close-up on hands and shoes, ...


  0%|          | 0/25 [00:00<?, ?it/s]

Saved: /content/drive/MyDrive/msc-deepfake/generated_videos/wan/w2_008_wan_20260719_165326.mp4 (245.4s = 4.1min)

Batch complete. Success: 8, Failed: 0
Total Wan videos now: 8/40


In [16]:
existing_ids = set()
for v in os.listdir(OUT_FOLDER):
    if v.endswith(".mp4"):
        prompt_id = v.split("_wan_")[0]
        existing_ids.add(prompt_id)

remaining = [line for line in lines if line.split("|")[0] not in existing_ids]

batch = remaining[:8]
print(f"Already done: {len(existing_ids)}")
print(f"Total remaining: {len(remaining)}")
print(f"This batch: {len(batch)}")
print("=" * 60)

successes = []
failures = []

for line in batch:
    parts = line.split("|", 2)
    if len(parts) != 3:
        continue
    prompt_id, category, prompt_text = parts
    try:
        generate_and_save(prompt_id, category, prompt_text, OUT_FOLDER, METADATA_FOLDER)
        successes.append(prompt_id)
    except Exception as e:
        print(f"FAILED on {prompt_id}: {e}")
        failures.append((prompt_id, str(e)))

now_done = len(existing_ids) + len(successes)
print("\n" + "=" * 60)
print(f"Batch complete. Success: {len(successes)}, Failed: {len(failures)}")
print(f"Total Wan videos now: {now_done}/40")
if failures:
    print(f"Failures: {failures}")

Already done: 8
Total remaining: 32
This batch: 8

[w2_009] hands: A person writing in a notebook with a fountain pen, close-up...


  0%|          | 0/25 [00:00<?, ?it/s]

Saved: /content/drive/MyDrive/msc-deepfake/generated_videos/wan/w2_009_wan_20260719_165735.mp4 (245.3s = 4.1min)

[w2_010] hands: Two hands shuffling a deck of playing cards on a wooden tabl...


  0%|          | 0/25 [00:00<?, ?it/s]

Saved: /content/drive/MyDrive/msc-deepfake/generated_videos/wan/w2_010_wan_20260719_170141.mp4 (245.4s = 4.1min)

[w2_011] multi_person: Two friends walking down a busy street talking to each other...


...

Saved: /content/drive/MyDrive/msc-deepfake/generated_videos/wan/w2_014_wan_20260719_171805.mp4 (245.4s = 4.1min)

[w2_015] multi_person: A couple sitting on a park bench feeding pigeons, autumn lea...


  0%|          | 0/25 [00:00<?, ?it/s]

Saved: /content/drive/MyDrive/msc-deepfake/generated_videos/wan/w2_015_wan_20260719_172211.mp4 (245.5s = 4.1min)

[w2_016] motion: A person kicking a soccer ball toward the camera, grass fiel...


  0%|          | 0/25 [00:00<?, ?it/s]

Saved: /content/drive/MyDrive/msc-deepfake/generated_videos/wan/w2_016_wan_20260719_172618.mp4 (245.4s = 4.1min)

Batch complete. Success: 8, Failed: 0
Total Wan videos now: 16/40


In [17]:
existing_ids = set()
for v in os.listdir(OUT_FOLDER):
    if v.endswith(".mp4"):
        prompt_id = v.split("_wan_")[0]
        existing_ids.add(prompt_id)

remaining = [line for line in lines if line.split("|")[0] not in existing_ids]

batch = remaining[:8]
print(f"Already done: {len(existing_ids)}")
print(f"Total remaining: {len(remaining)}")
print(f"This batch: {len(batch)}")
print("=" * 60)

successes = []
failures = []

for line in batch:
    parts = line.split("|", 2)
    if len(parts) != 3:
        continue
    prompt_id, category, prompt_text = parts
    try:
        generate_and_save(prompt_id, category, prompt_text, OUT_FOLDER, METADATA_FOLDER)
        successes.append(prompt_id)
    except Exception as e:
        print(f"FAILED on {prompt_id}: {e}")
        failures.append((prompt_id, str(e)))

now_done = len(existing_ids) + len(successes)
print("\n" + "=" * 60)
print(f"Batch complete. Success: {len(successes)}, Failed: {len(failures)}")
print(f"Total Wan videos now: {now_done}/40")
if failures:
    print(f"Failures: {failures}")

Already done: 16
Total remaining: 24
This batch: 8

[w2_017] motion: A jogger running along a beach at sunrise, waves in the back...


  0%|          | 0/25 [00:00<?, ?it/s]

Saved: /content/drive/MyDrive/msc-deepfake/generated_videos/wan/w2_017_wan_20260719_173114.mp4 (244.9s = 4.1min)

[w2_018] motion: A skateboarder performing a trick on a ramp, urban skate par...


  0%|          | 0/25 [00:00<?, ?it/s]

Saved: /content/drive/MyDrive/msc-deepfake/generated_videos/wan/w2_018_wan_20260719_173520.mp4 (245.2s = 4.1min)

[w2_019] motion: Water splashing as a swimmer dives into a pool, high-speed c...


...

Saved: /content/drive/MyDrive/msc-deepfake/generated_videos/wan/w2_022_wan_20260719_175144.mp4 (245.3s = 4.1min)

[w2_023] text_scene: A person walking past a chalkboard menu outside a cafe...


  0%|          | 0/25 [00:00<?, ?it/s]

Saved: /content/drive/MyDrive/msc-deepfake/generated_videos/wan/w2_023_wan_20260719_175550.mp4 (245.7s = 4.1min)

[w2_024] text_scene: A traveller checking a departure board at a train station...


  0%|          | 0/25 [00:00<?, ?it/s]

Saved: /content/drive/MyDrive/msc-deepfake/generated_videos/wan/w2_024_wan_20260719_175957.mp4 (245.7s = 4.1min)

Batch complete. Success: 8, Failed: 0
Total Wan videos now: 24/40


In [18]:
existing_ids = set()
for v in os.listdir(OUT_FOLDER):
    if v.endswith(".mp4"):
        prompt_id = v.split("_wan_")[0]
        existing_ids.add(prompt_id)

remaining = [line for line in lines if line.split("|")[0] not in existing_ids]

batch = remaining[:8]
print(f"Already done: {len(existing_ids)}")
print(f"Total remaining: {len(remaining)}")
print(f"This batch: {len(batch)}")
print("=" * 60)

successes = []
failures = []

for line in batch:
    parts = line.split("|", 2)
    if len(parts) != 3:
        continue
    prompt_id, category, prompt_text = parts
    try:
        generate_and_save(prompt_id, category, prompt_text, OUT_FOLDER, METADATA_FOLDER)
        successes.append(prompt_id)
    except Exception as e:
        print(f"FAILED on {prompt_id}: {e}")
        failures.append((prompt_id, str(e)))

now_done = len(existing_ids) + len(successes)
print("\n" + "=" * 60)
print(f"Batch complete. Success: {len(successes)}, Failed: {len(failures)}")
print(f"Total Wan videos now: {now_done}/40")
if failures:
    print(f"Failures: {failures}")

Already done: 24
Total remaining: 16
This batch: 8

[w2_025] text_scene: A student writing on a whiteboard in a classroom, equations ...


  0%|          | 0/25 [00:00<?, ?it/s]

Saved: /content/drive/MyDrive/msc-deepfake/generated_videos/wan/w2_025_wan_20260719_180423.mp4 (245.2s = 4.1min)

[w2_026] texture: Close-up of a chef chopping vegetables on a wooden cutting b...


  0%|          | 0/25 [00:00<?, ?it/s]

Saved: /content/drive/MyDrive/msc-deepfake/generated_videos/wan/w2_026_wan_20260719_180830.mp4 (245.4s = 4.1min)

[w2_027] texture: A weaver working on a traditional loom, colourful threads vi...


...

Saved: /content/drive/MyDrive/msc-deepfake/generated_videos/wan/w2_030_wan_20260719_182454.mp4 (245.3s = 4.1min)

[w2_031] animal: A golden retriever running in slow motion across a lawn, sun...


  0%|          | 0/25 [00:00<?, ?it/s]

Saved: /content/drive/MyDrive/msc-deepfake/generated_videos/wan/w2_031_wan_20260719_182900.mp4 (245.1s = 4.1min)

[w2_032] animal: A cat stretching lazily on a sunlit windowsill, indoor scene...


  0%|          | 0/25 [00:00<?, ?it/s]

Saved: /content/drive/MyDrive/msc-deepfake/generated_videos/wan/w2_032_wan_20260719_183306.mp4 (245.3s = 4.1min)

Batch complete. Success: 8, Failed: 0
Total Wan videos now: 32/40


In [19]:
existing_ids = set()
for v in os.listdir(OUT_FOLDER):
    if v.endswith(".mp4"):
        prompt_id = v.split("_wan_")[0]
        existing_ids.add(prompt_id)

remaining = [line for line in lines if line.split("|")[0] not in existing_ids]

batch = remaining[:8]
print(f"Already done: {len(existing_ids)}")
print(f"Total remaining: {len(remaining)}")
print(f"This batch: {len(batch)}")
print("=" * 60)

successes = []
failures = []

for line in batch:
    parts = line.split("|", 2)
    if len(parts) != 3:
        continue
    prompt_id, category, prompt_text = parts
    try:
        generate_and_save(prompt_id, category, prompt_text, OUT_FOLDER, METADATA_FOLDER)
        successes.append(prompt_id)
    except Exception as e:
        print(f"FAILED on {prompt_id}: {e}")
        failures.append((prompt_id, str(e)))

now_done = len(existing_ids) + len(successes)
print("\n" + "=" * 60)
print(f"Batch complete. Success: {len(successes)}, Failed: {len(failures)}")
print(f"Total Wan videos now: {now_done}/40")
if failures:
    print(f"Failures: {failures}")

Already done: 32
Total remaining: 8
This batch: 8

[w2_033] animal: A horse galloping through an open field, wind blowing its ma...


  0%|          | 0/25 [00:00<?, ?it/s]

Saved: /content/drive/MyDrive/msc-deepfake/generated_videos/wan/w2_033_wan_20260719_183755.mp4 (244.9s = 4.1min)

[w2_034] animal: A parrot preening its feathers on a wooden perch, tropical b...


  0%|          | 0/25 [00:00<?, ?it/s]

Saved: /content/drive/MyDrive/msc-deepfake/generated_videos/wan/w2_034_wan_20260719_184202.mp4 (245.7s = 4.1min)

[w2_035] animal: A school of fish swimming through a coral reef, underwater s...


...

Saved: /content/drive/MyDrive/msc-deepfake/generated_videos/wan/w2_038_wan_20260719_185825.mp4 (245.0s = 4.1min)

[w2_039] edge_case: A crowded market at night with many people walking, string l...


  0%|          | 0/25 [00:00<?, ?it/s]

Saved: /content/drive/MyDrive/msc-deepfake/generated_videos/wan/w2_039_wan_20260719_190231.mp4 (245.1s = 4.1min)

[w2_040] edge_case: A person walking their dog past a shop window, both dog and ...


  0%|          | 0/25 [00:00<?, ?it/s]

Saved: /content/drive/MyDrive/msc-deepfake/generated_videos/wan/w2_040_wan_20260719_190637.mp4 (245.2s = 4.1min)

Batch complete. Success: 8, Failed: 0
Total Wan videos now: 40/40
